In [1]:
import torch

from torch_openreml import MixedModelREML
from torch_openreml.covariance import DummyMatrix, ScalarMatrix

n, p = 10, 2

group = ["a", "b"] * 5
effect = torch.tensor([1.0, -1.0]).repeat(n // 2)

X = torch.randn(n, p)
y = X @ torch.tensor([0.5, -0.5]) + effect + 0.5 * torch.randn(n)

reml = MixedModelREML(
    genotype=(DummyMatrix(group), ScalarMatrix(group)),
)

names, free_params = reml.optimize(y, X, verbose=1)
names, free_params

[  1] ∥∇∥:       3.8991, ∥Δ∥: 0.9391, η: 0.50, ∥Δᶜ∥: 0.4696, accepted: True , log 𝓛:  -4.6835
[  2] ∥∇∥:       0.7302, ∥Δ∥: 0.0811, η: 1.00, ∥Δᶜ∥: 0.0811, accepted: True , log 𝓛:  -4.6574
[  3] ∥∇∥:       0.0495, ∥Δ∥: 0.0045, η: 1.00, ∥Δᶜ∥: 0.0045, accepted: True , log 𝓛:  -4.6573
[  4] ∥∇∥:       0.0002, ∥Δ∥: 0.0000, η: 0.50, ∥Δᶜ∥: 0.0000, accepted: True , log 𝓛:  -4.6573

[∇: score, Δ: 𝐉⁻¹∇, η: step length, Δᶜ: clip(𝛉 + ηΔ, lb, ub) - 𝛉, 𝓛: restricted likelihood]

✓ Converged at iteration 4, total time: 0.01s



(['G/sigma^2', 'R/sigma^2'], tensor([ 0.0626, -0.5187]))

In [2]:
history = reml.history
history["loglik"]

[tensor(-4.6835), tensor(-4.6574), tensor(-4.6573), tensor(-4.6573)]

In [3]:
torch.stack(history["loglik"])

tensor([-4.6835, -4.6574, -4.6573, -4.6573])

In [4]:
torch.stack(history["free_params"])

tensor([[ 0.1081, -0.4570],
        [ 0.0606, -0.5227],
        [ 0.0626, -0.5187],
        [ 0.0626, -0.5187]])

In [5]:
names, theta_hat = reml.get_free_params("last")
names, theta_hat

(['G/sigma^2', 'R/sigma^2'], tensor([ 0.0626, -0.5187]))

In [6]:
params = reml.build_params("last", out_format="dict")
params

{'G/sigma^2': tensor([1.1335]), 'R/sigma^2': tensor([0.3544])}

In [7]:
tree = reml.tree("last")
tree

{'/': tensor([[1.4878, 0.0000, 1.1335, 0.0000, 1.1335, 0.0000, 1.1335, 0.0000, 1.1335,
          0.0000],
         [0.0000, 1.4878, 0.0000, 1.1335, 0.0000, 1.1335, 0.0000, 1.1335, 0.0000,
          1.1335],
         [1.1335, 0.0000, 1.4878, 0.0000, 1.1335, 0.0000, 1.1335, 0.0000, 1.1335,
          0.0000],
         [0.0000, 1.1335, 0.0000, 1.4878, 0.0000, 1.1335, 0.0000, 1.1335, 0.0000,
          1.1335],
         [1.1335, 0.0000, 1.1335, 0.0000, 1.4878, 0.0000, 1.1335, 0.0000, 1.1335,
          0.0000],
         [0.0000, 1.1335, 0.0000, 1.1335, 0.0000, 1.4878, 0.0000, 1.1335, 0.0000,
          1.1335],
         [1.1335, 0.0000, 1.1335, 0.0000, 1.1335, 0.0000, 1.4878, 0.0000, 1.1335,
          0.0000],
         [0.0000, 1.1335, 0.0000, 1.1335, 0.0000, 1.1335, 0.0000, 1.4878, 0.0000,
          1.1335],
         [1.1335, 0.0000, 1.1335, 0.0000, 1.1335, 0.0000, 1.1335, 0.0000, 1.4878,
          0.0000],
         [0.0000, 1.1335, 0.0000, 1.1335, 0.0000, 1.1335, 0.0000, 1.1335, 0.0000,
    

In [8]:
beta_hat = reml.blue(y, X, theta_hat)
beta_hat

tensor([ 0.5867, -0.5962])

In [9]:
cov_beta = reml.cov_blue(X, theta_hat)
cov_beta

tensor([[ 0.0547, -0.0315],
        [-0.0315,  0.0625]])

In [10]:
se_beta = torch.sqrt(reml.var_blue(X, theta_hat))
se_beta

tensor([0.2338, 0.2500])

In [11]:
b_hat = reml.blup()
b_hat

tensor([ 0.8496, -1.1721])

In [12]:
b_cov = reml.cov_blup()
b_cov

tensor([[ 1.0557, -0.0071],
        [-0.0071,  1.0398]])

In [13]:
b_var = reml.var_blup()
b_var

tensor([1.0557, 1.0398])

In [14]:
b_by_term = reml.blup_per_term()
b_by_term

{'genotype': tensor([ 0.8496, -1.1721])}

In [15]:
b_cov_by_term = reml.cov_blup_per_term()
b_cov_by_term

{'genotype': tensor([[ 1.0557, -0.0071],
         [-0.0071,  1.0398]])}

In [16]:
b_var_by_term = reml.var_blup_per_term()
b_var_by_term

{'genotype': tensor([1.0557, 1.0398])}

In [17]:
y_marginal = reml.marginal_predict()
y_marginal

tensor([ 0.8279, -0.4634, -0.3210, -1.3299, -0.6576, -0.1625, -0.7833, -0.0503,
         0.0212, -0.3831])

In [18]:
y_conditional = reml.predict()
y_conditional

tensor([ 1.6774, -1.6354,  0.5286, -2.5020,  0.1920, -1.3346,  0.0663, -1.2224,
         0.8707, -1.5552])

In [19]:
y_new = reml.predict_new(
    X[:2],
    {"genotype": torch.tensor([[1.0, 0.0], [0.0, 1.0]])},
)
y_new

tensor([ 1.6774, -1.6354])

In [20]:
z_new = DummyMatrix(group).transform(["a", "b"])

y_new = reml.predict_new(
    X[:2],
    z_new,
)
y_new

tensor([ 1.6774, -1.6354])

In [21]:
e_marginal = reml.marginal_residual(y, X, theta_hat)
e_marginal

tensor([ 1.2346, -1.9717,  0.1149, -0.9028,  0.8648, -0.6585,  1.5057, -1.4496,
         0.7933, -1.2442])

In [22]:
e_conditional = reml.residual(y, X, theta_hat)
e_conditional

tensor([ 0.3851, -0.7996, -0.7346,  0.2692,  0.0153,  0.5136,  0.6562, -0.2776,
        -0.0562, -0.0721])

In [23]:
c11, c12, c21, c22 = reml.c_inverse_blocks(X, theta_hat)
c11, c12, c21, c22

(tensor([[ 0.0547, -0.0315],
         [-0.0315,  0.0625]]),
 tensor([[ 0.0217,  0.0305],
         [-0.0020, -0.0386]]),
 tensor([[ 0.0217, -0.0020],
         [ 0.0305, -0.0386]]),
 tensor([[0.0778, 0.0071],
         [0.0071, 0.0937]]))

In [24]:
loglik_value, loglik_method = reml.loglik(y, X, theta_hat)
loglik_value, loglik_method

(tensor(-4.6573), 'cform')

In [25]:
reml.is_converged()

True